# 🔧 Iceberg Table Maintenance

Iceberg tables accumulate snapshots, metadata files, and small data files over time. This notebook demonstrates the five core maintenance operations to keep your tables healthy, performant, and cost-efficient.

| Operation | Purpose |
|---|---|
| **Expire Snapshots** | Remove old snapshots to free up metadata & data file references |
| **Remove Old Metadata Files** | Delete stale `.metadata.json` files left by expired snapshots |
| **Delete Orphan Files** | Clean up data files not referenced by any snapshot |
| **Compact Data Files** | Merge small files into larger ones for better read performance |

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Connect to Trino

In [ ]:
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

---
## 🏗️ Setup — Create & Populate a Demo Table

We create a demo table and simulate several batches of inserts + updates. Each operation creates a new Iceberg snapshot, resulting in multiple snapshots, metadata files, and small data files — exactly the conditions that require maintenance.

In [ ]:
# Create demo table
run_query("""
CREATE OR REPLACE TABLE iceberg.bronze.demo_events (
    event_id  VARCHAR,
    user_id   BIGINT,
    action    VARCHAR,
    amount    DOUBLE,
    event_ts  TIMESTAMP(6) WITH TIME ZONE
) WITH (
    format = 'PARQUET',
    extra_properties = MAP(
        ARRAY[
            'write.metadata.delete-after-commit.enabled', 
            'write.metadata.previous-versions-max'
        ],
        ARRAY[
            'true', 
            '3'
        ]
    )
)
""")
print("✅ Table 'demo_events' created in iceberg.bronze")

In [ ]:
# Batch 1 — initial inserts (snapshot 1)
run_query("""
INSERT INTO iceberg.bronze.demo_events VALUES
    ('e001', 1, 'purchase', 29.99, TIMESTAMP '2026-01-01 10:00:00.000000 UTC'),
    ('e002', 2, 'purchase', 49.99, TIMESTAMP '2026-01-01 10:05:00.000000 UTC'),
    ('e003', 3, 'refund',   15.00, TIMESTAMP '2026-01-01 10:10:00.000000 UTC')
""")

# Batch 2 (snapshot 2)
run_query("""
INSERT INTO iceberg.bronze.demo_events VALUES
    ('e004', 4, 'purchase', 99.00, TIMESTAMP '2026-01-02 11:00:00.000000 UTC'),
    ('e005', 5, 'purchase', 19.99, TIMESTAMP '2026-01-02 11:05:00.000000 UTC')
""")

# Batch 3 (snapshot 3)
run_query("""
INSERT INTO iceberg.bronze.demo_events VALUES
    ('e006', 1, 'purchase', 59.99, TIMESTAMP '2026-01-03 09:00:00.000000 UTC'),
    ('e007', 2, 'refund',    5.00, TIMESTAMP '2026-01-03 09:15:00.000000 UTC'),
    ('e008', 6, 'purchase', 34.99, TIMESTAMP '2026-01-03 09:30:00.000000 UTC')
""")

# Batch 4 (snapshot 4)
run_query("""
INSERT INTO iceberg.bronze.demo_events VALUES
    ('e009', 7, 'purchase', 12.00, TIMESTAMP '2026-01-04 14:00:00.000000 UTC'),
    ('e010', 8, 'purchase', 74.99, TIMESTAMP '2026-01-04 14:20:00.000000 UTC')
""")

print("✅ 4 batches inserted — 4 snapshots created, 4 small data files on disk")

In [ ]:
# Inspect initial state
print("📸 Snapshot count:")
run_query('SELECT count(*) AS snapshot_count FROM iceberg.bronze."demo_events$snapshots"')
print()
print("📁 Data file count:")
run_query('SELECT count(*) AS file_count, sum(file_size_in_bytes) AS total_bytes FROM iceberg.bronze."demo_events$files"')

---
## 1️⃣ Expire Snapshots

Every write (INSERT, UPDATE, DELETE) creates an Iceberg **snapshot**. Old snapshots enable time travel but consume metadata storage. `expire_snapshots` removes snapshots older than the retention threshold, freeing up references to old data.

> ⚠️ **Trade-off:** Once a snapshot is expired, you can no longer time-travel to it. Keep a retention window long enough to support rollback needs (e.g. `7d` in production).

```
Before:  snap-1 → snap-2 → snap-3 → snap-4 (current)
After:   snap-4 (only current kept with retention_threshold='0s')
```

In [ ]:
print("📸 Before — snapshot history:")
run_query("""
SELECT committed_at, snapshot_id, operation
FROM iceberg.bronze."demo_events$snapshots"
ORDER BY committed_at
""")
print()

# Expire all snapshots older than now (keeps only the current one)
run_query("ALTER TABLE iceberg.bronze.demo_events EXECUTE expire_snapshots(retention_threshold => '0s')")

print("📸 After — snapshot history:")
run_query("""
SELECT committed_at, snapshot_id, operation
FROM iceberg.bronze."demo_events$snapshots"
ORDER BY committed_at
""")
print()
print("✅ Old snapshots expired — time travel to those points is no longer possible")

## 2️⃣ Remove Old Metadata Files

Every commit in Iceberg (writes, schema changes, property updates) generates a new `.metadata.json` file. While old snapshots are removed by `expire_snapshots`, the metadata files themselves persist unless explicitly managed.

Trino manages this via two properties:
1. `metadata_delete_after_commit`: Enables automatic deletion of old metadata files.
2. `metadata_previous_versions_to_keep`: Controls how many old metadata files to retain.

In this scenario, we'll observe the current metadata log, tighten the retention policy, and trigger a cleanup.

In [ ]:
# 1. Count current metadata files (accumulated from setup and previous operations)
print("📊 Current metadata file count:")
run_query('SELECT count(*) AS metadata_count FROM iceberg.bronze."demo_events$metadata_log_entries"')

print("📄 Listing current metadata files:")
run_query('SELECT file, timestamp FROM iceberg.bronze."demo_events$metadata_log_entries" ORDER BY timestamp DESC')
print()

In [ ]:
# 2. Tighten the retention to keep only the 2 most recent versions
print("🔧 Reducing 'metadata_previous_versions_to_keep' to 2...")
run_query("""
ALTER TABLE iceberg.bronze.demo_events
SET PROPERTIES extra_properties = MAP(
    ARRAY[
        'write.metadata.previous-versions-max'
    ],
    ARRAY[
        '2'
    ]
)
""")

# 3. Verify the log has been trimmed
print("📊 Metadata file count after update:")
run_query('SELECT count(*) AS metadata_count FROM iceberg.bronze."demo_events$metadata_log_entries"')

print("📄 Remaining metadata files (only 2 kept + the new one created by the ALTER command):")
run_query('SELECT file, timestamp FROM iceberg.bronze."demo_events$metadata_log_entries" ORDER BY timestamp DESC')
print("✅ Metadata files trimmed automatically upon commit.")

---
## 3️⃣ Delete Orphan Files

**Orphan files** are data or metadata files on storage that are not referenced by *any* Iceberg snapshot. They accumulate from:
- Failed or aborted writes (partial commits)
- Expired snapshots whose data files weren't immediately cleaned up
- Manual file manipulation

`remove_orphan_files` scans the table's storage location and deletes files not known to Iceberg's metadata.

> ⚠️ Use a conservative `retention_threshold` (e.g. `'7d'`) in production to avoid deleting files from in-flight writes.

In [ ]:
print("🔍 Scanning and removing orphan files...")
print("   (files not referenced by any snapshot will be deleted)")
print()

# Remove orphan files from the maintenance table
result = run_query(
    "ALTER TABLE iceberg.bronze.demo_events EXECUTE remove_orphan_files(retention_threshold => '0s')",
    display=False
)

if result:
    print(f"🗑️  {len(result)} orphan file(s) found and removed:")
    for row in result:
        print(f"   {row[0]}")
else:
    print("✅ No orphan files found — storage is clean")

---
## 4️⃣ Compact Data Files

Every INSERT creates at least one new **data file**. Frequent small inserts produce many tiny files which hurt read performance (more S3 GETs, more planning overhead). `optimize` (Trino's compaction procedure) merges small files up to the target size.

```
Before:  [file-1: 1KB] [file-2: 1KB] [file-3: 1KB] [file-4: 1KB]
After:   [file-merged: ~4KB]   (one larger, efficient file)
```

> 💡 Run compaction periodically (e.g. daily via Airflow) or after bulk loading. The `file_size_threshold` controls what counts as "small".

In [ ]:
print("📁 Before compaction — data files:")
run_query('SELECT count(*) AS file_count, sum(file_size_in_bytes) AS total_bytes FROM iceberg.bronze."demo_events$files"')
print()

# Compact files smaller than 10 MB into a single larger file
run_query("ALTER TABLE iceberg.bronze.demo_events EXECUTE optimize(file_size_threshold => '10MB')")

print("📁 After compaction — data files:")
run_query('SELECT count(*) AS file_count, sum(file_size_in_bytes) AS total_bytes FROM iceberg.bronze."demo_events$files"')
print()
print("✅ Compaction complete — fewer, larger files for efficient reads")

In [ ]:
# After compaction, a new snapshot is created — the old (pre-compaction) files
# become orphans. Expire the compaction snapshot and clean up the old files.
run_query("ALTER TABLE iceberg.bronze.demo_events EXECUTE expire_snapshots(retention_threshold => '0s')")
run_query(
    "ALTER TABLE iceberg.bronze.demo_events EXECUTE remove_orphan_files(retention_threshold => '0s')",
    display=False
)
print("✅ Post-compaction cleanup done — old small files removed")

---
## 📊 Summary

| Operation | Trino Command | When to Run | Trade-off |
|---|---|---|---|
| **Expire Snapshots** | `EXECUTE expire_snapshots` | Regularly (e.g. daily) | Disables time travel for expired snapshots |
| **Remove Metadata Files** | Set `metadata_previous_versions_to_keep` | Once, at table creation | None — background operation |
| **Delete Orphan Files** | `EXECUTE remove_orphan_files` | After failed writes or expire | Use conservative threshold to avoid deleting in-flight writes |
| **Compact Data Files** | `EXECUTE optimize` | After bulk loads / streaming ingestion | Creates a new snapshot; follow with expire + orphan cleanup |

### Recommended Maintenance Schedule

```
Daily:
  1. optimize(file_size_threshold => '128MB')      -- compact small files
  2. expire_snapshots(retention_threshold => '7d')  -- keep a week for rollback
  3. remove_orphan_files(retention_threshold => '1d') -- cleanup stragglers
```


---
## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to drop the demo table and schema:
# run_query("DROP TABLE IF EXISTS iceberg.maintenance.demo_events")
# run_query("DROP SCHEMA IF EXISTS iceberg.maintenance")
# print("🗑️ Demo table and schema dropped")